In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import random
import math

import pandas as pd

import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots
import fly_plot_lib.plot as fpl

import scipy.stats

In [ ]:
from splitflow import unifying_algo_analysis_helper as uaah
from splitflow.set_zorder_functions import *

# path lookup (flat local Data/unifying_algo_results/ or Dryad Data/Unifying_Algo_Results/{Main,Supplemental}/) and glyph-id fix
import slope_r2_helper as h

In [ ]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

In [ ]:
MARKERSIZE = 0.75
MODEL_LINE_COLOR = 'lightcoral'
TRANSLATION = True

In [ ]:
FIGURE_NAME = 'unifying_analysis.svg'

In [ ]:
if TRANSLATION:
    FIGURE_NAME = 'unifying_analysis_translationTrue.svg'

In [ ]:
def get_median_slope_and_axis_ratio(fname):
    df = pd.read_parquet(fname)

    slopes = []
    axis_ratios = []
    rotation = []
    rho = []

    for obj_id in df.obj_id_unique_event.unique():
        data = df[df.obj_id_unique_event==obj_id].copy()
        #slopes.append(data.slope.abs().median())

        data['slope_abs'] = data.slope.abs()
        slopes.append( uaa.get_weighted_value(data, col='slope_abs', use='rmse_affine') )

        axis_ratios.append( uaa.get_weighted_value(data, col='axis_ratio', use='rmse_affine') )

        data.rotation = data.rotation.abs()
        rotation.append( uaa.get_weighted_value(data, col='rotation', use='rmse_affine') )

        data.rho = data.rho.abs()
        rho.append( uaa.get_weighted_value(data, col='rho', use='rmse_affine') )

            
    results = {'slopes': np.array(slopes)/0.01,
               'axis_ratios': np.array(axis_ratios),
               'rotation': np.array(rotation),
               'rho': np.array(rho),
               #'percent_affine_improvements': np.array(percent_affine_improvements),
              }
    
    return results

In [ ]:
metadata = {
            h.unifying_results_path('stillair_data_flash_translation' + str(TRANSLATION) + '.parquet'): [1, '#084a72ff', 'stillair'], 
            h.unifying_results_path('unsteady_top_on_data_flash_translation' + str(TRANSLATION) + '.parquet'): [2, '#2b75b3ff', 'unsteady'], # 2b75b3b3 
            h.unifying_results_path('laminar_data_flash_translation' + str(TRANSLATION) + '.parquet'): [3, '#991128ff', 'laminar'], 
           }

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
ax = layout.axes[('flash_model', 'slopes')]

feature = 'slopes'

x_data_for_regression = []
y_data_for_regression = []

ticks = []
ticklabels = []

dpoints = []
for key, val in metadata.items():
    fname = key
    results = get_median_slope_and_axis_ratio(fname)
    xix, color, label = val

    ticks.append(xix)
    ticklabels.append(label)
    
    hist_counts, hist_bins = np.histogram(results[feature], bins=20)
    fpl.scatter_box_violin_distribution(ax, xix, copy.copy(results[feature]), hist_counts, hist_bins, color=color, markersize=MARKERSIZE)

    dpoints.append(results[feature])
    
    if xix<=3:
        x_data_for_regression.append(xix)
        y_data_for_regression.append(np.median(results[feature]))

print('median slope: ', np.median(y_data_for_regression))
median_slope = np.median(y_data_for_regression)
xixs = [1,2,3]
ax.hlines([median_slope,],
          xixs[0]-0.5, xixs[-1]+0.5, 
          linestyles=['--',], colors=[MODEL_LINE_COLOR,])

ax.set_xlim(0.5, 3.5)
ax.set_ylim(0,8)

ax.set_ylabel('Slope (rad/s)', labelpad=-2)


ax.set_xticks(ticks)
ax.set_xticklabels([])

ax.set_yticks([0, 2, 4, 6, 8])
ax.set_yticklabels(['0', '', '', '', '8'])

ax.tick_params(axis='y', pad=2)

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)

# data = diagnose_axis_elements(ax, verbose=True)
set_selective_rasterization(ax, rasterize_markers=['o'])

#ax.set_rasterization_zorder(1000)

# Kruskal-Wallis H test

In [ ]:
scipy.stats.kruskal(dpoints[0], dpoints[1],dpoints[2])

In [ ]:
ax = layout.axes[('flash_model', 'axis_ratios')]

feature = 'axis_ratios'

x_data_for_regression = []
y_data_for_regression = []

ticks = []
ticklabels = []

dpoints = []

for key, val in metadata.items():
    fname = key
    results = get_median_slope_and_axis_ratio(fname)
    xix, color, label = val

    ticks.append(xix)
    ticklabels.append(label)
    
    hist_counts, hist_bins = np.histogram(results[feature], bins=20)
    fpl.scatter_box_violin_distribution(ax, xix, copy.copy(results[feature]), hist_counts, hist_bins, color=color, markersize=MARKERSIZE)

    dpoints.append(results[feature]) 
    
    if xix<=3:
        x_data_for_regression.append(xix)
        y_data_for_regression.append(np.median(results[feature]))

result = scipy.stats.linregress(x_data_for_regression, y_data_for_regression)
axis_ratio_slope = result.slope
axis_ratio_intercept = result.intercept
x = np.linspace(0.5, 3.5, 10)
y = x*axis_ratio_slope + axis_ratio_intercept
ax.plot(x, y, '--', color=MODEL_LINE_COLOR)

print('axis ratios')
print(x_data_for_regression)
print(y_data_for_regression)

model_axis_ratio = {}
for key, val in metadata.items():
    fname = key
    results = get_median_slope_and_axis_ratio(fname)
    xix, color, label = val
    model_axis_ratio_for_condition = xix*axis_ratio_slope + axis_ratio_intercept
    model_axis_ratio.setdefault(fname, model_axis_ratio_for_condition)

print('axis_ratio_slope: ', axis_ratio_slope)
print('axis_ratio_intercept: ', axis_ratio_intercept)

ax.set_xlim(0.5, 3.5)
ax.set_ylim(0,1)

ax.set_ylabel('Axis ratio', labelpad=-2)

ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0', '', '', '', '', '1'])

ax.set_xticks(ticks)
ax.set_xticklabels([])

ax.tick_params(axis='y', pad=2)

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)
set_selective_rasterization(ax, rasterize_markers=['o'])

In [ ]:

"""
Paste this cell after the slope plotting cell (cell 61 in the notebook),
which ends with `set_selective_rasterization(ax, rasterize_markers=['o'])`.

It assumes `dpoints` is a list of 3 arrays: [laminar_slopes, unsteady_slopes, stillair_slopes],
which is built in that cell via `dpoints.append(results[feature])`.
"""

import numpy as np
import scipy.stats
from itertools import combinations

# =============================================================================
# Verify data is available
# =============================================================================
assert len(dpoints) == 3, f"Expected 3 conditions in dpoints, got {len(dpoints)}"
condition_labels = ['stillair', 'unsteady', 'laminar']

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
for i, label in enumerate(condition_labels):
    d = dpoints[i]
    print(f"  {label:>10s}: n={len(d)}, mean={np.mean(d):.3f}, median={np.median(d):.3f}, sd={np.std(d):.3f}")

# =============================================================================
# Jonckheere-Terpstra (ordered trend: laminar < unsteady < stillair)
# =============================================================================
def jonckheere_terpstra(groups):
    """
    Jonckheere-Terpstra test for an increasing trend across ordered groups.
    groups: list of arrays in hypothesized order.
    """
    k = len(groups)
    J = 0
    for i, j in combinations(range(k), 2):
        for xi in groups[i]:
            for xj in groups[j]:
                if xj > xi:
                    J += 1
                elif xj == xi:
                    J += 0.5

    n = [len(g) for g in groups]
    N = sum(n)
    E_J = (N**2 - sum(ni**2 for ni in n)) / 4
    Var_J = (N**2 * (2 * N + 3) - sum(ni**2 * (2 * ni + 3) for ni in n)) / 72

    Z = (J - E_J) / np.sqrt(Var_J)
    p_one = scipy.stats.norm.cdf(Z)
    p_two = 2 * min(p_one, 1 - p_one)

    return J, E_J, Z, p_one, p_two

J, E_J, Z, p_one, p_two = jonckheere_terpstra(dpoints)

print("\n" + "=" * 60)
print("Jonckheere-Terpstra (ordered: still air to laminar)")
print("=" * 60)
print(f"  J = {J:.1f}")
print(f"  E[J] = {E_J:.1f}")
print(f"  Z = {Z:.4f}")
print(f"  p (one-sided, increasing) = {p_one:.2e}")
print(f"  p (two-sided) = {p_two:.2e}")



# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Hypothesized order: still air to laminar")
print(f"  Group medians: {[f'{np.median(d):.3f}' for d in dpoints]}")
print(f"")
print(f"  Jonckheere-Terpstra p (one-sided): {p_one:.2e}")


In [ ]:
ax = layout.axes[('flash_model', 'rotations')]

feature = 'rotation'

ticks = []
ticklabels = []
for key, val in metadata.items():
    fname = key
    results = get_median_slope_and_axis_ratio(fname)
    xix, color, label = val

    ticks.append(xix)
    ticklabels.append(label)
    
    hist_counts, hist_bins = np.histogram(results[feature], bins=20)
    fpl.scatter_box_violin_distribution(ax, xix, copy.copy(results[feature]), hist_counts, hist_bins, color=color, markersize=MARKERSIZE)



ax.set_xlim(0.5, 3.5)
ax.set_ylim(0,np.pi)

ax.set_ylabel('Rotation angle', labelpad=-2)

ax.set_yticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
ax.set_yticklabels(['0', '', '', '', '$\pi$'])

ax.set_xticks(ticks)
ax.set_xticklabels([])

ax.tick_params(axis='y', pad=2)

fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                 tick_length=2.5,
                                 spine_locations={'left': 5, 'bottom': 5},
                                 linewidth=0.5)
fifi.mpl_functions.set_fontsize(ax, 6)
set_selective_rasterization(ax, rasterize_markers=['o'])

In [ ]:
layout.append_figure_to_layer(layout.figures['flash_model'], 'flash_model', cleartarget=True)
layout.write_svg(FIGURE_NAME)
# give this layer's text glyphs layer-specific ids, so rewriting another layer can't remove glyphs panel F uses
h.uniquify_glyph_ids(FIGURE_NAME, ['flash_model'])